# Bank Marketing Imbalanced Classification

This notebook presents a clean machine learning workflow for the Bank Marketing dataset. It starts by loading and inspecting the data, then builds models for the imbalanced classification problem.

The goal is to predict whether a client subscribed to a term deposit.

## 1. Import Libraries

In [ ]:
from pathlib import Path

import pandas as pd
from scipy.io.arff import loadarff

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.pipeline import Pipeline

## 2. Load the Dataset

In [ ]:
DATA_PATH = Path("phpkIxskf.arff")

COLUMN_NAMES = [
    "age", "job", "marital", "education", "default", "balance", "housing", "loan",
    "contact", "day", "month", "duration", "campaign", "pdays", "previous", "poutcome", "y"
]

raw_data, metadata = loadarff(DATA_PATH)
df = pd.DataFrame(raw_data)
df.columns = COLUMN_NAMES

for column in df.select_dtypes(include=["object"]).columns:
    df[column] = df[column].str.decode("utf-8")

target_map = {"1": 0, "2": 1, "no": 0, "yes": 1}
df["y"] = df["y"].map(target_map).astype(int)

df.head()

## 3. Inspect the Data

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe()

In [ ]:
df.describe(include="object")

## 4. Check the Target Imbalance

The positive class is much smaller than the negative class, so accuracy alone is not enough for evaluation.

In [ ]:
target_counts = df["y"].value_counts().rename(index={0: "no", 1: "yes"})
target_percentages = df["y"].value_counts(normalize=True).rename(index={0: "no", 1: "yes"}) * 100

pd.DataFrame({"count": target_counts, "percentage": target_percentages.round(2)})

## 5. Prepare Train and Test Sets

The split happens before any resampling. This avoids data leakage, because synthetic or duplicated samples must not influence the test set.

In [ ]:
DROP_DURATION = False

model_df = df.copy()

if DROP_DURATION:
    model_df = model_df.drop(columns=["duration"])

X = model_df.drop(columns=["y"])
y = model_df["y"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

X_train.shape, X_test.shape

## 6. Build the Preprocessing Pipeline

Numeric features are scaled. Categorical features are one-hot encoded instead of being dropped.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
    ]
)

numeric_features, categorical_features

## 7. Define Models

The resampling methods are placed inside pipelines, so they are applied only to the training data.

In [ ]:
logistic = LogisticRegression(max_iter=2000, random_state=42)
balanced_logistic = LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)
forest = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1,
)

experiments = {
    "Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", logistic),
    ]),
    "Balanced Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("model", balanced_logistic),
    ]),
    "SMOTE + Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("sampler", SMOTE(random_state=42, k_neighbors=3)),
        ("model", logistic),
    ]),
    "Random Oversampling + Logistic Regression": Pipeline([
        ("preprocess", preprocessor),
        ("sampler", RandomOverSampler(random_state=42)),
        ("model", logistic),
    ]),
    "Balanced Random Forest": Pipeline([
        ("preprocess", preprocessor),
        ("model", forest),
    ]),
}

## 8. Train and Evaluate

For imbalanced data, recall, precision, F1-score, ROC-AUC, and average precision are more informative than accuracy alone.

In [ ]:
def evaluate_model(name, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        y_pred,
        average="binary",
        zero_division=0,
    )

    print(f"\n{name}")
    print("-" * len(name))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, target_names=["no", "yes"], zero_division=0))

    return {
        "model": name,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc_score(y_test, y_score),
        "average_precision": average_precision_score(y_test, y_score),
    }


results = [evaluate_model(name, model) for name, model in experiments.items()]

## 9. Compare Results

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=["average_precision", "f1"],
    ascending=False,
)

results_df

## 10. Conclusion

This workflow is more reliable than resampling before the train-test split. The final comparison should be based mainly on the minority class performance, especially recall, F1-score, ROC-AUC, and average precision.

The `duration` column is kept because it is part of the dataset and useful for assignment analysis. For a real pre-call marketing prediction system, it should usually be removed because the call duration is only known after the call.